# Smart-Shelf · 07 · Build Executive Report (PDF)

**Final deliverable.** Compiles all findings from notebooks 01-06 into a single 9-page client-facing PDF using ReportLab.

## Setup — auto-resolving paths

Run this cell first. It finds the project root automatically.

In [ ]:


from pathlib import Path
import os

# Find smart_shelf root by walking up from current working directory
def find_project_root():
    p = Path.cwd().resolve()
    for parent in [p] + list(p.parents):
        if (parent / "scripts").exists() and (parent / "outputs").exists():
            return parent
        if parent.name == "smart_shelf":
            return parent
    # Fallback: assume notebook lives in smart_shelf/notebooks/
    return Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()

PROJECT_ROOT = find_project_root()
DATASET_DIR  = PROJECT_ROOT.parent / "dataset"   # sibling of smart_shelf/
DATA_DIR     = PROJECT_ROOT / "data"
OUTPUTS_DIR  = PROJECT_ROOT / "outputs"
FIGURES_DIR  = PROJECT_ROOT / "figures"

for d in [DATA_DIR, OUTPUTS_DIR, FIGURES_DIR]:
    d.mkdir(parents=True, exist_ok=True)

print(f"Project root : {PROJECT_ROOT}")
print(f"Dataset dir  : {DATASET_DIR}")
print(f"Data dir     : {DATA_DIR}")
print(f"Outputs dir  : {OUTPUTS_DIR}")
print(f"Figures dir  : {FIGURES_DIR}")

assert DATASET_DIR.exists(), f"Dataset folder not found at {DATASET_DIR}. Place Olist CSVs there."


In [ ]:
from datetime import datetime
from reportlab.lib.pagesizes import A4
from reportlab.lib.styles import getSampleStyleSheet, ParagraphStyle
from reportlab.lib.units import cm
from reportlab.lib.colors import HexColor
from reportlab.lib import colors
from reportlab.platypus import (SimpleDocTemplate, Paragraph, Spacer, Image,
                                 PageBreak, Table, TableStyle)

pdf_path = OUTPUTS_DIR / "Smart_Shelf_Executive_Report.pdf"
doc = SimpleDocTemplate(str(pdf_path), pagesize=A4,
                        leftMargin=2*cm, rightMargin=2*cm,
                        topMargin=1.8*cm, bottomMargin=1.8*cm)

# Colors
NAVY  = HexColor("#1F3864")
TEAL  = HexColor("#2E7D7D")
ORANGE= HexColor("#E07A1F")
GREY  = HexColor("#595959")

ss = getSampleStyleSheet()
styles = {
    "title"   : ParagraphStyle("title",   parent=ss["Title"], fontName="Helvetica-Bold",
                                fontSize=22, textColor=NAVY, spaceAfter=6, leading=26),
    "subtitle": ParagraphStyle("subtitle", parent=ss["Normal"], fontName="Helvetica-Oblique",
                                fontSize=12, textColor=GREY, spaceAfter=18, leading=15),
    "h1"      : ParagraphStyle("h1", parent=ss["Heading1"], fontName="Helvetica-Bold",
                                fontSize=16, textColor=NAVY, spaceBefore=16, spaceAfter=8, leading=20),
    "h2"      : ParagraphStyle("h2", parent=ss["Heading2"], fontName="Helvetica-Bold",
                                fontSize=13, textColor=TEAL, spaceBefore=10, spaceAfter=6, leading=16),
    "body"    : ParagraphStyle("body", parent=ss["BodyText"], fontName="Helvetica",
                                fontSize=10.5, leading=15, spaceAfter=6, alignment=4),
    "callout" : ParagraphStyle("callout", parent=ss["Normal"], fontName="Helvetica-Bold",
                                fontSize=11, textColor=ORANGE, leading=14, spaceAfter=8),
    "small"   : ParagraphStyle("small", parent=ss["Normal"], fontName="Helvetica-Oblique",
                                fontSize=8.5, textColor=GREY, leading=11),
}

story = []
print("Styles loaded — ready to build PDF")


## Cover + Executive Summary

In [ ]:
# COVER
story.append(Spacer(1, 4*cm))
story.append(Paragraph("Smart-Shelf", styles["title"]))
story.append(Paragraph("Logistics &amp; Margin Optimisation Engine", styles["title"]))
story.append(Paragraph(f"Diagnostic report prepared for OmniStore "
                       f"&nbsp;·&nbsp; {datetime.now().strftime('%B %Y')}",
                       styles["subtitle"]))
story.append(Spacer(1, 1*cm))
story.append(Paragraph("<b>Dataset:</b> Olist Brazilian E-commerce · "
                       "112,650 order items · 99K orders · 3K sellers · 32K products · 2016-2018",
                       styles["body"]))
story.append(Spacer(1, 0.5*cm))
story.append(Paragraph("<b>Brief:</b> Build an automated diagnostic engine that identifies margin "
                       "leaks by correlating supply chain inefficiencies with pricing strategies.",
                       styles["body"]))
story.append(PageBreak())

# EXEC SUMMARY
story.append(Paragraph("Executive Summary", styles["h1"]))
story.append(Paragraph(
    "OmniStore's margin paradox — surging online sales but shrinking margins — is real and "
    "quantifiable. After analysing 112,650 order items, we find:", styles["body"]))

bullets = [
    ("40.7% of active SKUs", "have freight-to-price ratios above 30%, burning <b>R$ 465K</b> in freight on flagged products."),
    ("153 SKUs are loss-makers", "where freight cost actually exceeds product price. Worst at <b>657% freight ratio</b>."),
    ("OTIF rate of 81.7%", "is 8-13 percentage points below the 90-95% industry benchmark, with last-mile delivery consuming <b>73.6% of cycle time</b>."),
    ("Late delivery is the dominant churn driver", " — the low-review rate rises from 8.8% (early) to <b>75.1% (>7 days late)</b>."),
    ("A 2-day Lead Time reduction yields ~R$ 178K margin uplift", " primarily through stockout-cancellation prevention."),
]
data_rows = [[Paragraph(f"<b>{a}</b>", styles["body"]),
              Paragraph(b, styles["body"])] for a, b in bullets]
t = Table(data_rows, colWidths=[6*cm, 10*cm])
t.setStyle(TableStyle([
    ("VALIGN", (0,0), (-1,-1), "TOP"),
    ("LEFTPADDING", (0,0), (-1,-1), 4),
    ("RIGHTPADDING", (0,0), (-1,-1), 4),
    ("TOPPADDING", (0,0), (-1,-1), 4),
    ("BOTTOMPADDING", (0,0), (-1,-1), 4),
    ("LINEBELOW", (0,0), (-1,-2), 0.5, GREY),
]))
story.append(t)
story.append(Spacer(1, 0.4*cm))

story.append(Paragraph("Recommended actions", styles["h2"]))
story.append(Paragraph(
    "1. <b>Re-price or re-route</b> the 1,968 flagged heavy-freight SKUs.<br/>"
    "2. <b>Delist</b> the 153 true loss-makers unless strategically retained.<br/>"
    "3. <b>Concentrate last-mile investment</b> in northeast Brazil (worst-OTIF states: AL, MA, CE).<br/>"
    "4. <b>Implement bullwhip dampening</b> — current operating profile amplifies a 10% demand shift into a 40% supplier-tier swing.<br/>"
    "5. <b>Target 2-day Lead Time reduction</b> in Year-1 — projected ~R$ 178K uplift.",
    styles["body"]))
story.append(PageBreak())

print("Cover + Executive Summary built")


## Task pages 1-4

In [ ]:
# TASK 1
story.append(Paragraph("Task 1 · Data Architecture &amp; Lead Time Variance", styles["h1"]))
story.append(Paragraph(
    "All nine Olist tables were merged at the order-item grain into a single 112,650-row analytical "
    "fact table (43 engineered columns). Lead time variance was computed as the difference between "
    "actual delivery and estimated delivery; positive values indicate lateness.", styles["body"]))

story.append(Paragraph("Lead time variance directly drives stockout risk", styles["h2"]))
lead_data = [
    ["Variance bucket", "Orders", "Stockout-risk rate", "Avg freight (R$)"],
    ["≤ -10 days (very early)", "65,852", "4.9%",  "20.66"],
    ["-10 to -5 days",          "25,657", "10.7%", "18.02"],
    ["-5 to 0 days",            "9,965",  "18.2%", "18.32"],
    ["0 to 5 days late",        "4,055",  "28.7%", "20.29"],
    ["5 to 10 days late",       "2,139",  "30.2%", "22.74"],
    ["> 10 days late",          "2,477",  "26.4%", "24.50"],
]
t = Table(lead_data, colWidths=[5.5*cm, 3*cm, 4*cm, 4*cm])
t.setStyle(TableStyle([
    ("BACKGROUND", (0,0), (-1,0), NAVY),
    ("TEXTCOLOR",  (0,0), (-1,0), colors.white),
    ("FONTNAME",   (0,0), (-1,0), "Helvetica-Bold"),
    ("FONTSIZE",   (0,0), (-1,-1), 9.5),
    ("ALIGN",      (1,1), (-1,-1), "RIGHT"),
    ("ROWBACKGROUNDS", (0,1), (-1,-1), [colors.white, HexColor("#F5F5F5")]),
    ("GRID",       (0,0), (-1,-1), 0.3, colors.lightgrey),
]))
story.append(t)
story.append(Spacer(1, 0.3*cm))
story.append(Paragraph(
    "<b>Reading:</b> stockout risk rises monotonically from 4.9% on very-early deliveries to 30.2% "
    "on orders 5-10 days late. Lead-time discipline is therefore not a soft KPI — it is an "
    "inventory-availability lever.", styles["callout"]))
story.append(PageBreak())

# TASK 2
story.append(Paragraph("Task 2 · OTIF &amp; Cycle Time Decomposition", styles["h1"]))
story.append(Paragraph(
    "Traditional OEE — designed for manufacturing equipment — is replaced by the e-commerce equivalent: "
    "OTIF (On-Time In-Full) and Cycle Time Decomposition.", styles["body"]))

story.append(Paragraph("OTIF rates", styles["h2"]))
otif_data = [
    ["Metric", "Value", "Industry benchmark"],
    ["On-Time rate", "92.09%", "≥ 95%"],
    ["In-Full rate (review-proxied)", "85.34%", "≥ 95%"],
    ["OTIF (combined)", "81.66%", "90-95%"],
]
t = Table(otif_data, colWidths=[7*cm, 4.5*cm, 5*cm])
t.setStyle(TableStyle([
    ("BACKGROUND", (0,0), (-1,0), NAVY), ("TEXTCOLOR", (0,0), (-1,0), colors.white),
    ("FONTNAME", (0,0), (-1,0), "Helvetica-Bold"),
    ("FONTNAME", (0,3), (-1,3), "Helvetica-Bold"),
    ("BACKGROUND", (0,3), (-1,3), HexColor("#FFF2CC")),
    ("FONTSIZE", (0,0), (-1,-1), 10),
    ("ALIGN", (1,1), (-1,-1), "CENTER"),
    ("GRID", (0,0), (-1,-1), 0.3, colors.lightgrey),
]))
story.append(t)
story.append(Spacer(1, 0.4*cm))

story.append(Paragraph("Cycle Time Decomposition", styles["h2"]))
ct_data = [
    ["Stage", "Mean (d)", "Median (d)", "% of cycle", "CV"],
    ["Stage 1 — Purchase → Approval",        "0.44", "0.01", "3.5%",  "1.98"],
    ["Stage 2 — Approval → Carrier Handoff", "2.85", "1.84", "22.8%", "1.26"],
    ["Stage 3 — Carrier → Customer",         "9.19", "7.06", "73.6%", "0.94"],
]
t = Table(ct_data, colWidths=[6*cm, 2.5*cm, 2.5*cm, 2.5*cm, 2*cm])
t.setStyle(TableStyle([
    ("BACKGROUND", (0,0), (-1,0), NAVY), ("TEXTCOLOR", (0,0), (-1,0), colors.white),
    ("FONTNAME", (0,0), (-1,0), "Helvetica-Bold"),
    ("BACKGROUND", (0,3), (-1,3), HexColor("#FCE4D6")),
    ("FONTSIZE", (0,0), (-1,-1), 9.5),
    ("ALIGN", (1,1), (-1,-1), "RIGHT"),
    ("GRID", (0,0), (-1,-1), 0.3, colors.lightgrey),
]))
story.append(t)
story.append(Spacer(1, 0.4*cm))
story.append(Paragraph(
    "<b>Diagnosis: MECHANICAL.</b> Stage 3 (carrier handoff to customer delivery) consumes 73.6% of "
    "total cycle time. Investment in last-mile — alternative carriers, hub redistribution, regional "
    "fulfilment centres — will deliver the largest cycle-time reduction.", styles["callout"]))
story.append(PageBreak())

print("Tasks 1-2 built")


In [ ]:
# TASK 3
story.append(Paragraph("Task 3 · Bullwhip Effect Analysis", styles["h1"]))
story.append(Paragraph(
    "Modelled in <b>Bullwhip_Simulator.xlsx</b>. The model uses a 4-tier supply chain "
    "(retailer → distributor → manufacturer → supplier) with realistic lead times (1, 2, 3, 4 months) "
    "and a safety stock factor of 1.5×. At the brief's specified default — a 10% consumer demand "
    "shift — the model reproduces the textbook 4× bullwhip amplification:", styles["body"]))

bw_data = [
    ["Tier",         "Lead time (mo)", "Order qty", "% vs baseline", "Cumulative amplification"],
    ["Retailer",     "1", "622",   "12.8%", "1.27×"],
    ["Distributor",  "2", "653",   "18.4%", "1.84×"],
    ["Manufacturer", "3", "703",   "27.3%", "2.73×"],
    ["Supplier",     "4", "773",   "40.0%", "4.00×"],
]
t = Table(bw_data, colWidths=[3.2*cm, 2.8*cm, 2.5*cm, 3.2*cm, 4.5*cm])
t.setStyle(TableStyle([
    ("BACKGROUND", (0,0), (-1,0), NAVY), ("TEXTCOLOR", (0,0), (-1,0), colors.white),
    ("FONTNAME", (0,0), (-1,0), "Helvetica-Bold"),
    ("BACKGROUND", (0,4), (-1,4), HexColor("#FCE4D6")),
    ("FONTSIZE", (0,0), (-1,-1), 9.5),
    ("ALIGN", (1,1), (-1,-1), "CENTER"),
    ("GRID", (0,0), (-1,-1), 0.3, colors.lightgrey),
]))
story.append(t)
story.append(Spacer(1, 0.4*cm))
story.append(Paragraph(
    "A 10% consumer demand shift cascades into a <b>40% supplier-order shift</b> — exactly the "
    "amplification described in the brief. The Excel sensitivity grid shows that lowering the "
    "safety-stock factor from 1.5× to 1.0× cuts supplier volatility from 40% to 29% for the same "
    "10% consumer shift.", styles["callout"]))
story.append(PageBreak())

# TASK 4
story.append(Paragraph("Task 4 · Shipping Delays vs Customer Churn", styles["h1"]))
story.append(Paragraph(
    "The seaborn heatmap below correlates each customer's worst delivery experience (rows) with "
    "three churn-proxy metrics (columns).", styles["body"]))

img_path = FIGURES_DIR / "delay_churn_heatmap.png"
if img_path.exists():
    story.append(Image(str(img_path), width=15.5*cm, height=8*cm))
story.append(Spacer(1, 0.3*cm))

story.append(Paragraph(
    "<b>Headline:</b> the low-review rate (≤ 2 stars) climbs from 8.8% on very-early deliveries to "
    "<b>75.1% on orders > 7 days late</b> — an 8.5× jump. Late delivery is therefore the dominant "
    "driver of customer dissatisfaction.", styles["callout"]))
story.append(PageBreak())

print("Tasks 3-4 built")


## Deliverables 1 & 2 + Methodology

In [ ]:
# DELIVERABLE 1
story.append(Paragraph("Deliverable 1 · Automated Margin Report", styles["h1"]))
story.append(Paragraph(
    "The Python notebook <b>04_automated_margin_report.ipynb</b> applies three configurable rules "
    "to flag SKUs whose freight cost erodes margin:", styles["body"]))

rule_data = [
    ["Rule", "Definition", "SKUs flagged"],
    ["Loss-makers", "freight_value > price",                "153"],
    ["Heavy freight (default)", "freight / price > 30%",   "1,968"],
    ["Bottom-20% by category", "worst freight ratio in cat","1,010"],
]
t = Table(rule_data, colWidths=[4*cm, 8*cm, 4*cm])
t.setStyle(TableStyle([
    ("BACKGROUND", (0,0), (-1,0), NAVY), ("TEXTCOLOR", (0,0), (-1,0), colors.white),
    ("FONTNAME", (0,0), (-1,0), "Helvetica-Bold"),
    ("FONTSIZE", (0,0), (-1,-1), 10),
    ("ALIGN", (2,1), (2,-1), "RIGHT"),
    ("GRID", (0,0), (-1,-1), 0.3, colors.lightgrey),
]))
story.append(t)
story.append(Spacer(1, 0.4*cm))

story.append(Paragraph("Top 5 margin-leak categories (by freight burden)", styles["h2"]))
cat_data = [
    ["Category", "Freight burden (R$)", "Avg ratio"],
    ["garden_tools",      "48,891", "52%"],
    ["furniture_decor",   "39,771", "55%"],
    ["bed_bath_table",    "37,791", "51%"],
    ["telephony",         "33,815", "61%"],
    ["housewares",        "33,486", "65%"],
]
t = Table(cat_data, colWidths=[6*cm, 5*cm, 4*cm])
t.setStyle(TableStyle([
    ("BACKGROUND", (0,0), (-1,0), NAVY), ("TEXTCOLOR", (0,0), (-1,0), colors.white),
    ("FONTNAME", (0,0), (-1,0), "Helvetica-Bold"),
    ("FONTSIZE", (0,0), (-1,-1), 10),
    ("ALIGN", (1,1), (-1,-1), "RIGHT"),
    ("GRID", (0,0), (-1,-1), 0.3, colors.lightgrey),
]))
story.append(t)
story.append(Spacer(1, 0.3*cm))

img2 = FIGURES_DIR / "freight_ratio_distribution.png"
if img2.exists():
    story.append(Image(str(img2), width=15*cm, height=7.5*cm))
story.append(PageBreak())

# DELIVERABLE 2
story.append(Paragraph("Deliverable 2 · Supply Chain Simulator", styles["h1"]))
story.append(Paragraph(
    "The Excel file <b>Supply_Chain_Simulator.xlsx</b> models how reducing average lead time impacts "
    "total margin via three loss mechanisms.", styles["body"]))

uplift_data = [
    ["Days reduced", "New late-rate", "New stockout-rate",
     "Saving 1 (refunds)", "Saving 2 (stockouts)", "Saving 3 (LTV)", "Total uplift"],
    ["1", "7.41%", "8.72%", "R$ 6,611",  "R$ 79,329",  "R$ 2,900",  "R$ 88,839"],
    ["2", "6.91%", "8.12%", "R$ 13,221", "R$ 158,658", "R$ 5,799",  "R$ 177,678"],
    ["3", "6.41%", "7.52%", "R$ 19,832", "R$ 237,987", "R$ 8,699",  "R$ 266,518"],
    ["4", "5.91%", "6.92%", "R$ 26,443", "R$ 317,316", "R$ 11,598", "R$ 355,357"],
    ["5", "5.41%", "6.32%", "R$ 33,054", "R$ 396,645", "R$ 14,498", "R$ 444,196"],
]
t = Table(uplift_data, colWidths=[1.8*cm, 1.8*cm, 2.3*cm, 2.3*cm, 2.3*cm, 1.8*cm, 2.5*cm])
t.setStyle(TableStyle([
    ("BACKGROUND", (0,0), (-1,0), NAVY), ("TEXTCOLOR", (0,0), (-1,0), colors.white),
    ("FONTNAME", (0,0), (-1,0), "Helvetica-Bold"),
    ("FONTNAME", (-1,1), (-1,-1), "Helvetica-Bold"),
    ("BACKGROUND", (-1,1), (-1,-1), HexColor("#E2EFDA")),
    ("FONTSIZE", (0,0), (-1,-1), 8.5),
    ("ALIGN", (1,1), (-1,-1), "RIGHT"),
    ("GRID", (0,0), (-1,-1), 0.3, colors.lightgrey),
]))
story.append(t)
story.append(Spacer(1, 0.4*cm))
story.append(Paragraph(
    "Stockout-cancellation prevention is the dominant lever (≈89% of total uplift). The uplift "
    "scales linearly because elasticities are constant in this model.", styles["callout"]))
story.append(PageBreak())

# METHODOLOGY
story.append(Paragraph("Methodology &amp; Limitations", styles["h1"]))
story.append(Paragraph(
    "This report is fully reproducible from the seven notebooks and the public Olist dataset. "
    "Three honest caveats:", styles["body"]))

caveats = [
    ("In-Full proxy", "Olist provides no per-line backorder flag, so 'In-Full' is proxied by "
                       "review_score ≥ 3 (no major complaint)."),
    ("Stockout-risk proxy", "Inventory levels are not in the dataset. Stockout-risk is proxied by "
                             "seller breaches of shipping_limit_date."),
    ("Elasticities in Simulator", "The Lead-Time-to-Margin elasticities are calibrated estimates "
                                   "derived from Olist's variance distribution. A controlled A/B "
                                   "test would tighten these."),
]
for title, text in caveats:
    story.append(Paragraph(f"<b>{title}</b>", styles["h2"]))
    story.append(Paragraph(text, styles["body"]))

# BUILD
doc.build(story)
print(f"PDF saved: {pdf_path}")
print(f"File size: {pdf_path.stat().st_size / 1024:.0f} KB")


🎉 **All 7 notebooks complete!** Your full Smart-Shelf project is ready:\n\n- `data/master_orders.parquet` — master fact table\n- `outputs/Smart_Shelf_Executive_Report.pdf` — 9-page client report\n- `outputs/Bullwhip_Simulator.xlsx` — Task 3 deliverable\n- `outputs/Supply_Chain_Simulator.xlsx` — Deliverable 2\n- `outputs/margin_leak_report.csv` — full SKU-level flag table\n- `figures/*.png` — all charts